# Strategy 06 â€” SDFVG Scalping (Supply/Demand + Fair Value Gap + BOS) â€” Bybit M5

Based on the ForexFactory thread: **M5 Scalping Strategy â€” SDFVG**

## Core Concepts

| Concept | Description |
|---------|-------------|
| **Fair Value Gap (FVG)** | Triple-candle pattern: large middle candle whose body is not fully overlapped by prev-candle high and next-candle low. The gap between those wicks is the FVG. |
| **Supply/Demand Zone** | Origin area where price aggressively departed. Demand = bullish origin, Supply = bearish origin. |
| **Break of Structure (BOS)** | Price takes out a prior swing high (bullish BOS) or swing low (bearish BOS), confirming trend direction. |

## Long Entry Rules
1. Identify a **demand zone** (aggressive bullish departure).
2. Find an **FVG** at the **beginning** of the move away from the zone.
3. Confirm the zone caused a **break of structure** (higher high).
4. Wait for price to **pull back** into the demand zone.
5. Enter long if price has **not breached** the demand zone bottom.
6. SL = bottom of demand zone; TP = 1:2 RR (or target next market structure).

*(Inverse logic for shorts â€” supply zone + bearish BOS)*

## Advanced Filters (optional)
- **Clean zone** â€” price never returned after creation â†’ stronger reaction expected.
- **Early FVG** â€” FVG earlier in the departure leg â†’ larger reversal potential.
- **200 EMA** â€” take longs above 200 EMA, shorts below.
- **NY Session** â€” prefer trades before 12:00 NY time.

In [5]:
# SECTION 1 â€” Imports and parameters
import warnings
warnings.filterwarnings("ignore")

import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use("seaborn-v0_8-darkgrid")

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
except ImportError:
    pass

# â”€â”€ User parameters â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SYMBOL          = "BTCUSDT"
TF_ENTRY        = "M5"           # entry/signal timeframe
TF_TREND        = "H1"           # optional trend filter (200 EMA)

# Zone detection
ZONE_LOOKBACK       = 20         # bars to scan for swing high/low (demand/supply origin)
IMPULSE_ATR_MULT    = 1.5        # min candle body relative to ATR(14) to qualify as impulse
ATR_PERIOD          = 14

# FVG detection
FVG_MIN_GAP_TICKS   = 2.0       # minimum FVG size in ticks (1 tick = pip size)

# BOS confirmation
BOS_LOOKBACK        = 30         # bars to look back for prior swing used in BOS check

# Entry / risk
RR                  = 2.0        # reward-to-risk ratio
PENDING_EXPIRY_BARS = 12         # M5 bars before pending order expires (~60 min)

START_BALANCE       = 10_000.0
RISK_PER_TRADE      = 0.01       # 1% of balance per trade
COMMISSION_RATE     = 0.00055    # Bybit taker per side

# 200 EMA filter
USE_EMA200_FILTER   = True       # True = only take longs above 200 EMA, shorts below

# NY session filter (UTC hours; NY open = 13:30 UTC, 12:00 NY = 17:00 UTC)
USE_NY_SESSION      = False      # True = only trade 13:30â€“17:00 UTC

BARS_M5  = 8_000
BARS_H1  = 2_000

PIP_SIZE = 1.0   # BTCUSDT â€” $1 per tick

print(f"Symbol={SYMBOL}  TF_ENTRY={TF_ENTRY}  TF_TREND={TF_TREND}")
print(f"EMA200 filter={USE_EMA200_FILTER}  NY session filter={USE_NY_SESSION}")
print(f"RR={RR}  Risk/trade={RISK_PER_TRADE*100}%  Commission={COMMISSION_RATE*100:.4f}% per side")

Symbol=BTCUSDT  TF_ENTRY=M5  TF_TREND=H1
EMA200 filter=True  NY session filter=False
RR=2.0  Risk/trade=1.0%  Commission=0.0550% per side


In [6]:
# SECTION 2 â€” Load OHLCV from local cache
DATA_DIR = _ROOT / "notebooks" / "data"
_TEHRAN  = "Asia/Tehran"


def _load_csv(symbol: str, tf: str, limit: int) -> pd.DataFrame:
    csv_path = DATA_DIR / symbol / tf / "ohlcv.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Cache missing: {csv_path}\nRun 00_data_fetching_bybit.ipynb first."
        )
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    if df.index.tz is None:
        df.index = df.index.tz_localize(_TEHRAN)
    df = df.sort_index()
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    df = df[["open", "high", "low", "close", "volume"]]
    if limit and len(df) > limit:
        df = df.iloc[-limit:]
    return df


print(f"Loading {TF_ENTRY} candles ...")
m5 = _load_csv(SYMBOL, TF_ENTRY, limit=BARS_M5)
print(f"  -> {len(m5):,} rows | last: {m5.index[-1]}")

print(f"Loading {TF_TREND} candles ...")
h1 = _load_csv(SYMBOL, TF_TREND, limit=BARS_H1)
print(f"  -> {len(h1):,} rows | last: {h1.index[-1]}")

display(m5.tail(3))
display(h1.tail(3))

Loading M5 candles ...
  -> 8,000 rows | last: 2026-05-14 19:40:00+03:30
Loading H1 candles ...
  -> 1,999 rows | last: 2026-05-14 18:30:00+03:30


,open,high,low,close,volume
time,,,,,
2026-05-14 19:30:00+03:30,81244.0,81462.0,81244.0,81285.3,1353.665
2026-05-14 19:35:00+03:30,81285.3,81444.4,81285.3,81424.5,369.072
2026-05-14 19:40:00+03:30,81424.5,81547.2,81416.7,81507.6,546.509


,open,high,low,close,volume
time,,,,,
2026-05-14 16:30:00+03:30,79724.0,80035.8,79622.0,79656.0,4229.640
2026-05-14 17:30:00+03:30,79656.0,80975.9,79515.5,80922.1,9290.666
2026-05-14 18:30:00+03:30,80922.1,81295.5,80743.3,81244.0,6601.728


In [7]:
# SECTION 3 â€” Indicators: ATR, 200 EMA, and helpers

def add_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    df = df.copy()
    high, low, close = df["high"], df["low"], df["close"]
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    df["atr"] = tr.ewm(span=period, adjust=False).mean()
    return df


def add_ema200(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ema200"] = df["close"].ewm(span=200, adjust=False).mean()
    return df


def body_size(df: pd.DataFrame) -> pd.Series:
    return (df["close"] - df["open"]).abs()


m5 = add_atr(m5, ATR_PERIOD)
m5 = add_ema200(m5)
h1 = add_atr(h1, ATR_PERIOD)
h1 = add_ema200(h1)

print("ATR and EMA200 added.")
display(m5[["close", "atr", "ema200"]].tail(5))

ATR and EMA200 added.


,close,atr,ema200
time,,,
2026-05-14 19:20:00+03:30,81243.7,173.771485,79836.718185
2026-05-14 19:25:00+03:30,81244.0,160.268621,79850.720989
2026-05-14 19:30:00+03:30,81285.3,167.966138,79864.995407
2026-05-14 19:35:00+03:30,81424.5,166.783986,79880.512866
2026-05-14 19:40:00+03:30,81507.6,161.946121,79896.702788


In [8]:
# SECTION 4 â€” Core detectors: FVG, Demand/Supply zone, BOS

def detect_fvg(df: pd.DataFrame, min_gap: float = 0.0) -> pd.DataFrame:
    """
    For each candle i (the middle candle), check if:
      - Bullish FVG: prev_candle.high < next_candle.low  (gap above prev, below next)
      - Bearish FVG: prev_candle.low  > next_candle.high (gap below prev, above next)
    Returns a DataFrame with columns: fvg_type ('bull'/'bear'/'none'),
    fvg_top, fvg_bot (price boundaries of the gap).
    """
    n = len(df)
    fvg_type = ["none"] * n
    fvg_top  = [np.nan] * n
    fvg_bot  = [np.nan] * n

    highs  = df["high"].values
    lows   = df["low"].values

    for i in range(1, n - 1):
        gap_bull = lows[i + 1]  - highs[i - 1]   # positive = bullish gap
        gap_bear = lows[i - 1]  - highs[i + 1]   # positive = bearish gap

        if gap_bull > min_gap:
            fvg_type[i] = "bull"
            fvg_top[i]  = lows[i + 1]
            fvg_bot[i]  = highs[i - 1]
        elif gap_bear > min_gap:
            fvg_type[i] = "bear"
            fvg_top[i]  = lows[i - 1]
            fvg_bot[i]  = highs[i + 1]

    out = df[["open","high","low","close"]].copy()
    out["fvg_type"] = fvg_type
    out["fvg_top"]  = fvg_top
    out["fvg_bot"]  = fvg_bot
    return out


def is_impulse_candle(df: pd.DataFrame, i: int, atr_mult: float) -> bool:
    """True if candle i is an impulse: body >= atr_mult أ— ATR."""
    body = abs(df["close"].iloc[i] - df["open"].iloc[i])
    return body >= atr_mult * df["atr"].iloc[i]


def find_demand_zone(df: pd.DataFrame, anchor_bar: int, lookback: int) -> dict | None:
    """
    Look backward from anchor_bar for the last bullish impulse candle.
    The demand zone is [impulse.low, impulse.open] (base of the departure candle).
    Returns dict with zone_top, zone_bot, origin_bar, or None.
    """
    start = max(0, anchor_bar - lookback)
    for i in range(anchor_bar - 1, start - 1, -1):
        row = df.iloc[i]
        if row["close"] > row["open"] and is_impulse_candle(df, i, IMPULSE_ATR_MULT):
            return {
                "zone_top":   float(row["open"]),
                "zone_bot":   float(row["low"]),
                "origin_bar": i,
            }
    return None


def find_supply_zone(df: pd.DataFrame, anchor_bar: int, lookback: int) -> dict | None:
    """
    Look backward from anchor_bar for the last bearish impulse candle.
    The supply zone is [impulse.open, impulse.high].
    """
    start = max(0, anchor_bar - lookback)
    for i in range(anchor_bar - 1, start - 1, -1):
        row = df.iloc[i]
        if row["close"] < row["open"] and is_impulse_candle(df, i, IMPULSE_ATR_MULT):
            return {
                "zone_top":   float(row["high"]),
                "zone_bot":   float(row["open"]),
                "origin_bar": i,
            }
    return None


def bos_bullish(df: pd.DataFrame, origin_bar: int, current_bar: int, lookback: int) -> bool:
    """
    True if the highest high in [origin_bar, current_bar) was broken
    by a close above it after origin_bar â€” confirming bullish BOS.
    """
    start = max(0, origin_bar - lookback)
    prior_high = df["high"].iloc[start:origin_bar].max() if origin_bar > start else 0.0
    # Check if any close between origin_bar and current_bar exceeded prior_high
    closes_after = df["close"].iloc[origin_bar:current_bar]
    return bool((closes_after > prior_high).any())


def bos_bearish(df: pd.DataFrame, origin_bar: int, current_bar: int, lookback: int) -> bool:
    """
    True if the lowest low in [origin_bar, current_bar) was broken
    by a close below it after origin_bar â€” confirming bearish BOS.
    """
    start = max(0, origin_bar - lookback)
    prior_low = df["low"].iloc[start:origin_bar].min() if origin_bar > start else np.inf
    closes_after = df["close"].iloc[origin_bar:current_bar]
    return bool((closes_after < prior_low).any())


def fvg_exists_after_origin(fvg_df: pd.DataFrame, origin_bar: int,
                             current_bar: int, fvg_dir: str) -> bool:
    """
    True if at least one FVG of direction fvg_dir exists between origin_bar and
    origin_bar + (current_bar - origin_bar) // 2   (early in the leg).
    """
    leg_half = origin_bar + max(1, (current_bar - origin_bar) // 2)
    window   = fvg_df.iloc[origin_bar : leg_half]
    return (window["fvg_type"] == fvg_dir).any()


# Pre-compute FVG table for M5
fvg_m5 = detect_fvg(m5, min_gap=FVG_MIN_GAP_TICKS * PIP_SIZE)
bull_fvgs = fvg_m5[fvg_m5["fvg_type"] == "bull"]
bear_fvgs = fvg_m5[fvg_m5["fvg_type"] == "bear"]

print(f"Total FVGs detected on {TF_ENTRY}: {len(fvg_m5[fvg_m5['fvg_type'] != 'none'])}")
print(f"  Bull FVGs: {len(bull_fvgs)}")
print(f"  Bear FVGs: {len(bear_fvgs)}")
display(bull_fvgs[["fvg_type","fvg_top","fvg_bot"]].tail(5))

Total FVGs detected on M5: 1767
  Bull FVGs: 900
  Bear FVGs: 867


,fvg_type,fvg_top,fvg_bot
time,,,
2026-05-14 18:20:00+03:30,bull,80805.7,80687.7
2026-05-14 18:25:00+03:30,bull,80922.0,80830.0
2026-05-14 19:10:00+03:30,bull,80954.2,80923.0
2026-05-14 19:15:00+03:30,bull,81023.6,80974.9
2026-05-14 19:20:00+03:30,bull,81223.0,81033.3


In [9]:
# SECTION 5 â€” Signal generator (SDFVG setup scanner)

def is_ny_session(ts) -> bool:
    utc = ts.tz_convert("UTC")
    minutes = utc.hour * 60 + utc.minute
    return 13 * 60 + 30 <= minutes <= 17 * 60


def generate_signals(
    df: pd.DataFrame,
    fvg_df: pd.DataFrame,
    *,
    zone_lookback: int,
    bos_lookback: int,
    use_ema200: bool,
    use_ny_session: bool,
) -> pd.DataFrame:
    signals = []
    for i in range(max(zone_lookback, bos_lookback) + 1, len(df) - 1):
        row   = df.iloc[i]
        ts    = df.index[i]
        close = float(row["close"])
        ema200 = float(row["ema200"])

        if use_ny_session and not is_ny_session(ts):
            continue

        # LONG
        if not use_ema200 or close > ema200:
            zone = find_demand_zone(df, anchor_bar=i, lookback=zone_lookback)
            if zone is not None:
                origin, ztop, zbot = zone["origin_bar"], zone["zone_top"], zone["zone_bot"]
                if zbot <= close <= ztop and close > zbot:
                    if (fvg_exists_after_origin(fvg_df, origin, i, "bull")
                            and bos_bullish(df, origin, i, bos_lookback)):
                        dist = close - zbot
                        if dist > 0:
                            signals.append({
                                "bar": i, "time": ts, "side": "buy",
                                "entry": close, "sl": zbot, "tp": close + RR * dist,
                                "zone_top": ztop, "zone_bot": zbot,
                                "origin_bar": origin,
                                "qty": round((START_BALANCE * RISK_PER_TRADE) / dist, 6),
                            })

        # SHORT
        if not use_ema200 or close < ema200:
            zone = find_supply_zone(df, anchor_bar=i, lookback=zone_lookback)
            if zone is not None:
                origin, ztop, zbot = zone["origin_bar"], zone["zone_top"], zone["zone_bot"]
                if zbot <= close <= ztop and close < ztop:
                    if (fvg_exists_after_origin(fvg_df, origin, i, "bear")
                            and bos_bearish(df, origin, i, bos_lookback)):
                        dist = ztop - close
                        if dist > 0:
                            signals.append({
                                "bar": i, "time": ts, "side": "sell",
                                "entry": close, "sl": ztop, "tp": close - RR * dist,
                                "zone_top": ztop, "zone_bot": zbot,
                                "origin_bar": origin,
                                "qty": round((START_BALANCE * RISK_PER_TRADE) / dist, 6),
                            })

    return pd.DataFrame(signals)


print("Generating SDFVG signals ...")
signals_df = generate_signals(
    m5, fvg_m5,
    zone_lookback=ZONE_LOOKBACK,
    bos_lookback=BOS_LOOKBACK,
    use_ema200=USE_EMA200_FILTER,
    use_ny_session=USE_NY_SESSION,
)

print(f"Total signals: {len(signals_df)}")
if not signals_df.empty:
    print(f"  Long : {(signals_df['side']=='buy').sum()}")
    print(f"  Short: {(signals_df['side']=='sell').sum()}")
    display(signals_df[["time","side","entry","sl","tp","zone_top","zone_bot"]].tail(8))

Generating SDFVG signals ...
Total signals: 20
  Long : 9
  Short: 11


,time,side,entry,sl,tp,zone_top,zone_bot
12,2026-05-01 18:20:00+03:30,buy,78268.5,78260.6,78284.3,78302.5,78260.6
13,2026-05-05 09:35:00+03:30,buy,80886.2,80852.1,80954.4,80894.1,80852.1
14,2026-05-05 09:45:00+03:30,buy,80857.7,80852.1,80868.9,80894.1,80852.1
15,2026-05-05 10:15:00+03:30,buy,80884.0,80852.1,80947.8,80894.1,80852.1
16,2026-05-05 10:20:00+03:30,buy,80886.3,80852.1,80954.7,80894.1,80852.1
17,2026-05-12 05:15:00+03:30,sell,81113.0,81143.8,81051.4,81143.8,81110.5
18,2026-05-12 06:15:00+03:30,sell,81137.6,81143.8,81125.2,81143.8,81110.5
19,2026-05-14 08:15:00+03:30,sell,79399.9,79404.7,79390.3,79404.7,79397.7


In [10]:
# SECTION 6 â€” Backtest engine

def run_backtest(
    df: pd.DataFrame,
    signals: pd.DataFrame,
    *,
    start_balance: float,
    risk_per_trade: float,
    pending_expiry_bars: int,
    commission_rate: float,
) -> tuple[pd.DataFrame, pd.Series]:
    if signals.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    df        = df.copy().sort_index()
    sig_iter  = iter(signals.itertuples())
    trades    = []
    balance   = float(start_balance)
    equity_pts: list[tuple] = []

    pending:  dict | None = None
    position: dict | None = None
    next_sig  = next(sig_iter, None)

    for i in range(len(df)):
        idx = df.index[i]
        row = df.iloc[i]

        # equity snapshot
        floating = 0.0
        if position:
            if position["side"] == "buy":
                floating = (row["close"] - position["entry"]) * position["qty"]
            else:
                floating = (position["entry"] - row["close"]) * position["qty"]
        equity_pts.append((idx, balance + floating))

        # check open position
        if position is not None:
            hit_sl = hit_tp = False
            if position["side"] == "buy":
                hit_sl = row["low"]  <= position["sl"]
                hit_tp = row["high"] >= position["tp"]
            else:
                hit_sl = row["high"] >= position["sl"]
                hit_tp = row["low"]  <= position["tp"]

            if hit_sl or hit_tp:
                exit_price = position["sl"] if hit_sl else position["tp"]
                gross = (exit_price - position["entry"]) * position["qty"]
                if position["side"] == "sell":
                    gross = -gross
                comm = (position["entry"] + exit_price) * position["qty"] * commission_rate
                net  = gross - comm
                balance += net
                trades.append({
                    "entry_time":    position["entry_time"],
                    "exit_time":     idx,
                    "side":          position["side"],
                    "entry":         position["entry"],
                    "sl":            position["sl"],
                    "tp":            position["tp"],
                    "exit":          exit_price,
                    "qty":           position["qty"],
                    "gross_pnl":     round(gross, 4),
                    "commission":    round(comm, 4),
                    "pnl":           round(net, 4),
                    "balance_after": round(balance, 4),
                    "result":        "win" if net > 0 else "loss",
                })
                position = None
            else:
                continue  # holding position, don't open new signals

        # check pending order
        if pending is not None:
            age = i - pending["created_i"]
            if age > pending_expiry_bars:
                pending = None
            else:
                triggered = (
                    (pending["side"] == "buy"  and row["high"] >= pending["entry"]) or
                    (pending["side"] == "sell" and row["low"]  <= pending["entry"])
                )
                if triggered:
                    qty = (balance * risk_per_trade) / abs(pending["entry"] - pending["sl"])
                    position = {
                        "side":       pending["side"],
                        "entry":      pending["entry"],
                        "sl":         pending["sl"],
                        "tp":         pending["tp"],
                        "qty":        round(qty, 6),
                        "entry_time": idx,
                    }
                    pending = None
                    next_sig = next(sig_iter, None)
                    continue

        # load next signal when bar index matches
        if pending is None and position is None:
            while next_sig is not None and next_sig.bar <= i:
                sig = next_sig
                next_sig = next(sig_iter, None)
                if sig.bar == i:
                    pending = {
                        "side":      sig.side,
                        "entry":     sig.entry,
                        "sl":        sig.sl,
                        "tp":        sig.tp,
                        "created_i": i,
                    }
                    break

    trades_df = pd.DataFrame(trades)
    eq        = pd.Series({t: v for t, v in equity_pts}).sort_index()
    return trades_df, eq


print("Running backtest ...")
trades_df, equity_curve = run_backtest(
    m5, signals_df,
    start_balance=START_BALANCE,
    risk_per_trade=RISK_PER_TRADE,
    pending_expiry_bars=PENDING_EXPIRY_BARS,
    commission_rate=COMMISSION_RATE,
)

print(f"Total trades executed: {len(trades_df)}")
if not trades_df.empty:
    display(trades_df.tail(8))

Running backtest ...
Total trades executed: 10


,entry_time,exit_time,side,entry,sl,tp,exit,qty,gross_pnl,commission,pnl,balance_after,result
2,2026-04-24 04:45:00+03:30,2026-04-24 04:50:00+03:30,buy,78119.7,78116.6,78125.9,78116.6,23.437584,-72.6565,2013.9908,-2086.6473,5179.0036,loss
3,2026-04-28 11:35:00+03:30,2026-04-28 11:40:00+03:30,sell,76744.5,76746.9,76739.7,76746.9,21.579182,-51.7900,1821.7204,-1873.5104,3305.4932,loss
4,2026-04-30 09:10:00+03:30,2026-04-30 09:15:00+03:30,sell,75647.1,75648.8,75643.7,75648.8,19.444078,-33.0549,1617.9951,-1651.0500,1654.4432,loss
5,2026-04-30 14:10:00+03:30,2026-04-30 14:15:00+03:30,buy,76034.6,76033.4,76037.0,76033.4,13.787026,-16.5444,1153.1110,-1169.6554,484.7877,loss
6,2026-05-01 18:25:00+03:30,2026-05-01 18:30:00+03:30,buy,78268.5,78260.6,78284.3,78260.6,0.613655,-4.8479,52.8302,-57.6781,427.1097,loss
7,2026-05-05 09:50:00+03:30,2026-05-05 09:55:00+03:30,buy,80857.7,80852.1,80868.9,80852.1,0.762696,-4.2711,67.8345,-72.1056,355.0041,loss
8,2026-05-05 10:25:00+03:30,2026-05-05 10:30:00+03:30,buy,80886.3,80852.1,80954.7,80852.1,0.103802,-3.5500,9.2338,-12.7839,342.2202,loss
9,2026-05-12 06:20:00+03:30,2026-05-12 06:25:00+03:30,sell,81137.6,81143.8,81125.2,81143.8,0.551968,-3.4222,49.2658,-52.6880,289.5323,loss


In [11]:
# SECTION 7 â€” Metrics

def max_drawdown_pct(equity) -> float:
    eq   = pd.Series(equity).astype(float).dropna()
    if len(eq) < 2:
        return 0.0
    peak = eq.cummax()
    dd   = (eq - peak) / peak.replace(0, np.nan)
    return float(abs(dd.min()) * 100.0)


def summarize(trades: pd.DataFrame, start_balance: float, equity=None) -> pd.DataFrame:
    if trades.empty:
        return pd.DataFrame([{"trades": 0, "win_rate_%": 0, "net_pnl": 0,
                               "profit_factor": None, "max_drawdown_%": 0,
                               "return_%": 0}])
    wins   = trades[trades["pnl"] > 0]["pnl"]
    losses = trades[trades["pnl"] < 0]["pnl"]
    gp = wins.sum() if not wins.empty else 0.0
    gl = abs(losses.sum()) if not losses.empty else 0.0
    pf = gp / gl if gl > 0 else np.nan
    end_b = float(trades["balance_after"].iloc[-1])
    return pd.DataFrame([{
        "trades":           int(len(trades)),
        "win_rate_%":       round((trades["pnl"] > 0).mean() * 100, 2),
        "gross_pnl":        round(trades["gross_pnl"].sum(), 2),
        "total_commission": round(trades["commission"].sum(), 2),
        "net_pnl":          round(trades["pnl"].sum(), 2),
        "avg_net_pnl":      round(trades["pnl"].mean(), 4),
        "profit_factor":    round(float(pf), 3) if pd.notna(pf) else np.nan,
        "max_drawdown_%":   round(max_drawdown_pct(equity), 2),
        "start_balance":    round(start_balance, 2),
        "end_balance":      round(end_b, 2),
        "return_%":         round(((end_b / start_balance) - 1) * 100, 4),
    }])


if not trades_df.empty:
    trades_df["cumulative_net_pnl"] = trades_df["pnl"].cumsum()

summary = summarize(trades_df, START_BALANCE, equity_curve)
display(summary)

if not trades_df.empty:
    print("\nPer-side breakdown:")
    display(
        trades_df.groupby("side").agg(
            trades=("pnl", "count"),
            win_rate=("pnl", lambda x: round((x > 0).mean(), 3)),
            net_pnl=("pnl", "sum"),
            gross_pnl=("gross_pnl", "sum"),
            commission=("commission", "sum"),
        ).round(4)
    )

,trades,win_rate_%,gross_pnl,total_commission,net_pnl,avg_net_pnl,profit_factor,max_drawdown_%,start_balance,end_balance,return_%
0,10,0.0,-366.69,9343.78,-9710.47,-971.0468,0.0,97.1,10000.0,289.53,-97.1047



Per-side breakdown:


,trades,win_rate,net_pnl,gross_pnl,commission
side,,,,,
buy,5,0.0,-3398.8703,-101.8699,3297.0003
sell,5,0.0,-6311.5975,-264.8180,6046.7795


In [ ]:
# SECTION 8 — Charts

fig, axes = plt.subplots(3, 1, figsize=(17, 14),
                         gridspec_kw={"height_ratios": [4, 1, 1]})
ax_price, ax_eq, ax_pnl = axes

ax_price.plot(m5.index, m5["close"], color="steelblue", linewidth=0.7,
              label=f"{SYMBOL} M5 Close")
ax_price.plot(m5.index, m5["ema200"], color="orange", linewidth=1.0,
              linestyle="--", label="EMA 200", alpha=0.8)

# Shade last demand/supply zones from signals
if not signals_df.empty:
    seen_zones: set = set()
    for _, sig in signals_df.tail(60).iterrows():
        key = (round(sig["zone_top"], 1), round(sig["zone_bot"], 1))
        if key in seen_zones:
            continue
        seen_zones.add(key)
        color = "green" if sig["side"] == "buy" else "red"
        ax_price.axhspan(sig["zone_bot"], sig["zone_top"], alpha=0.07, color=color)

if not trades_df.empty:
    buys  = trades_df[trades_df["side"] == "buy"]
    sells = trades_df[trades_df["side"] == "sell"]
    ax_price.scatter(buys["entry_time"],  buys["entry"],
                     marker="^", color="lime",    s=55, zorder=5, label="Long entry")
    ax_price.scatter(sells["entry_time"], sells["entry"],
                     marker="v", color="tomato",  s=55, zorder=5, label="Short entry")
    wins_mask = trades_df["pnl"] > 0
    ax_price.scatter(trades_df[wins_mask]["exit_time"],  trades_df[wins_mask]["exit"],
                     marker="o", color="lime",    s=25, zorder=4, label="Win exit")
    ax_price.scatter(trades_df[~wins_mask]["exit_time"], trades_df[~wins_mask]["exit"],
                     marker="x", color="darkred", s=35, zorder=4, label="Loss exit")

ax_price.set_title(
    f"{SYMBOL} {TF_ENTRY} — SDFVG Strategy  |  "
    f"FVG + Supply/Demand + BOS  |  RR={RR}  |  EMA200={USE_EMA200_FILTER}"
)
ax_price.set_ylabel("Price")
ax_price.legend(loc="upper left", fontsize=8, ncol=2)

if equity_curve is not None and len(equity_curve) > 0:
    ax_eq.plot(equity_curve.index, equity_curve.values, color="purple", linewidth=1.0)
ax_eq.axhline(START_BALANCE, color="gray", linestyle="--", linewidth=0.8)
ax_eq.set_title("Equity Curve (net of commission)")
ax_eq.set_ylabel("Equity ($)")

if not trades_df.empty:
    colors = ["green" if p > 0 else "red" for p in trades_df["pnl"]]
    ax_pnl.bar(range(len(trades_df)), trades_df["pnl"], color=colors, alpha=0.75)
    ax_pnl.axhline(0, color="gray", linewidth=0.8)
ax_pnl.set_title("Net PnL per Trade")
ax_pnl.set_ylabel("Net PnL ($)")
ax_pnl.set_xlabel("Trade #")

plt.tight_layout()
plt.show()


In [ ]:
# SECTION 9 — Save results
RESULT_DIR = Path("./results") / "strategy06_sdfvg" / SYMBOL / TF_ENTRY
RESULT_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(RESULT_DIR / "metrics.csv", index=False)

if trades_df is None or trades_df.empty:
    pd.DataFrame(columns=[
        "entry_time","exit_time","side","entry","sl","tp","exit",
        "qty","gross_pnl","commission","pnl","balance_after",
        "result","cumulative_net_pnl"
    ]).to_csv(RESULT_DIR / "trades.csv", index=False)
else:
    trades_df.to_csv(RESULT_DIR / "trades.csv", index=False)

if not signals_df.empty:
    signals_df.to_csv(RESULT_DIR / "signals.csv", index=False)

print(f"Saved -> {RESULT_DIR}")
print(f"  metrics.csv  ({len(summary)} row)")
print(f"  trades.csv   ({len(trades_df)} rows)")
print(f"  signals.csv  ({len(signals_df)} rows)")
